# Recalculate velocities

In [38]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import pyarrow as pa
import pyarrow.parquet as pq
import gc
import psutil
import os
import geopandas as gpd
import shutil

In [41]:
# ============================================================
# Paths
# ============================================================

INPUT_DIR = Path(
    "/home/lukasscharf/projects/ADUCAT/data/EGMS/vienna"
)


OUTPUT_DIR = Path(
    "/home/lukasscharf/projects/ADUCAT/data/EGMS/vienna/processed"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Temporary chunks
temp_dir = OUTPUT_DIR / "temp_chunks"

temp_dir.mkdir(
    parents=True,
    exist_ok=True
)


In [26]:
# ============================================================
# Input settings
# ============================================================

ORBIT = "073"
PRODUCT_START = "2020"
PRODUCT_END = "2024"

# ============================================================
# Find input files
# ============================================================

PRODUCT_RANGE = f"{PRODUCT_START}_{PRODUCT_END}"

csv_files = sorted(
    f for f in INPUT_DIR.glob("*.csv")
    if (
        f"_{ORBIT}_" in f.name
        and f"_{PRODUCT_RANGE}_" in f.name
    )
)

print(
    f"Found {len(csv_files)} CSV files "
    f"for orbit {ORBIT}, "
    f"product period {PRODUCT_RANGE}"
)

for f in csv_files:
    print(f"  {f.name}")

Found 5 CSV files for orbit 073, product period 2020_2024
  EGMS_L2a_073_0285_IW1_VV_2020_2024_1.csv
  EGMS_L2a_073_0285_IW2_VV_2020_2024_1.csv
  EGMS_L2a_073_0286_IW1_VV_2020_2024_1.csv
  EGMS_L2a_073_0286_IW2_VV_2020_2024_1.csv
  EGMS_L2a_073_0287_IW1_VV_2020_2024_1.csv


In [27]:
# Final output
final_output = (
    OUTPUT_DIR /
    f"EGMS_{ORBIT}_{PRODUCT_RANGE}.parquet"
)

print("Final file name:", final_output)

Final file name: /home/lukasscharf/projects/ADUCAT/data/EGMS/vienna/processed/EGMS_073_2020_2024.parquet


### Functions

In [28]:
def add_velocity(gdf, intervals):
    """
    Calculate linear velocity and velocity standard error
    for EGMS time-series data.

    EGMS time-series columns are named YYYYMMDD, e.g.
    20220101, 20220113, ...

    The regression is identical to the Augmenterra implementation:

        np.polyfit(t, y, 1, cov=True)

    Velocity is the fitted slope in mm/year.
    """

    # --------------------------------------------------
    # Find EGMS time-series columns
    # --------------------------------------------------

    ts_cols = [
        c for c in gdf.columns
        if re.fullmatch(r"\d{8}", str(c))
    ]

    if not ts_cols:
        raise ValueError(
            "No EGMS time-series columns found."
        )

    dates = pd.to_datetime(
        ts_cols,
        format="%Y%m%d"
    )

    # --------------------------------------------------
    # Process each interval
    # --------------------------------------------------

    for start_date, end_date in intervals:

        print(
            f"    Processing "
            f"{start_date} → {end_date}"
        )

        start_date = pd.Timestamp(start_date)
        end_date = pd.Timestamp(end_date)

        # --------------------------------------------------
        # Select observations in this interval
        # --------------------------------------------------

        mask = (
            (dates >= start_date) &
            (dates <= end_date)
        )

        selected_cols = [
            c
            for c, keep in zip(ts_cols, mask)
            if keep
        ]

        selected_dates = dates[mask]

        if len(selected_dates) < 3:

            print(
                "      Skipping: "
                "fewer than 3 observations"
            )

            continue

        print(
            f"      Using {len(selected_dates)} observations "
            f"({selected_cols[0]} → {selected_cols[-1]})"
        )

        # --------------------------------------------------
        # Time in years
        # --------------------------------------------------

        t = (
            selected_dates - selected_dates[0]
        ).total_seconds() / (
            365.25 * 24 * 3600
        )

        t = np.asarray(
            t,
            dtype=np.float64
        )

        # --------------------------------------------------
        # Output arrays
        # --------------------------------------------------

        velocities = np.full(
            len(gdf),
            np.nan,
            dtype=np.float64
        )

        velocity_std = np.full(
            len(gdf),
            np.nan,
            dtype=np.float64
        )

        # --------------------------------------------------
        # Process every point in this chunk
        # --------------------------------------------------

        for local_i, (_, row) in enumerate(
            gdf.iterrows()
        ):

            y = pd.to_numeric(
                row[selected_cols],
                errors="coerce"
            ).to_numpy(
                dtype=np.float64
            )

            valid = np.isfinite(y)

            if valid.sum() >= 3:

                try:

                    coefficients, covariance = np.polyfit(
                        t[valid],
                        y[valid],
                        1,
                        cov=True
                    )

                    # --------------------------------------
                    # Mean linear velocity [mm/year]
                    # --------------------------------------

                    velocities[local_i] = round(
                        coefficients[0],
                        1
                    )

                    # --------------------------------------
                    # Standard error of velocity [mm/year]
                    # --------------------------------------

                    velocity_std[local_i] = round(
                        np.sqrt(covariance[0, 0]),
                        1
                    )

                except (np.linalg.LinAlgError, ValueError):
                    pass

            del y
            del valid

        # --------------------------------------------------
        # Add result columns
        # --------------------------------------------------

        start_str = start_date.strftime("%Y%m%d")
        end_str = end_date.strftime("%Y%m%d")

        vel_col = f"VEL_{start_str}_{end_str}"
        std_col = f"V_STDERR_{start_str}_{end_str}"

        new_columns = pd.DataFrame(
            {
                vel_col: velocities,
                std_col: velocity_std
            },
            index=gdf.index
        )

        gdf = pd.concat(
            [gdf, new_columns],
            axis=1
        )

        # --------------------------------------------------
        # Clean interval-specific memory
        # --------------------------------------------------

        del velocities
        del velocity_std
        del selected_cols
        del selected_dates
        del mask
        del t

        gc.collect()

    # --------------------------------------------------
    # Clean function-level objects
    # --------------------------------------------------

    del ts_cols
    del dates

    gc.collect()

    return gdf

In [29]:
# ============================================================
# Determine date columns
# ============================================================

def get_date_columns(csv_file, start_date, end_date):

    columns = pd.read_csv(
        csv_file,
        nrows=0
    ).columns

    date_columns = [
        col for col in columns
        if re.fullmatch(r"\d{8}", str(col))
        and start_date <= str(col) <= end_date
    ]

    return date_columns

In [30]:
process = psutil.Process(os.getpid())


def print_memory(label=""):

    rss_gb = (
        process.memory_info().rss
        / (1024 ** 3)
    )

    print(
        f"[MEMORY] {label}: "
        f"{rss_gb:.2f} GB RSS"
    )

### Processing

In [31]:
intervals = [
    ("2022-01-01", "2024-12-31"),
    ("2020-01-01", "2024-12-31")
]


CHUNK_SIZE = 20000

In [59]:

csv_files = sorted(
    f for f in INPUT_DIR.glob("*.csv")
    if (
        f"_{ORBIT}_" in f.name
        and f"_{PRODUCT_START}_{PRODUCT_END}_" in f.name
    )
)

print(
    f"Found {len(csv_files)} EGMS CSV files "
    f"for orbit {ORBIT}, "
    f"product period {PRODUCT_START}_{PRODUCT_END}"
)


Found 5 EGMS CSV files for orbit 073, product period 2020_2024


In [61]:
# ============================================================
# TEST: first and last file only
# ============================================================
if len(csv_files) < 2:
    raise ValueError(
        "Need at least two CSV files for this test."
    )

csv_files = [
    csv_files[0],
    csv_files[-1]
]

print()
print(f"Selected {len(csv_files)} CSV files:")

for f in csv_files:
    print(f"  {f.name}")


Selected 2 CSV files:
  EGMS_L2a_073_0285_IW1_VV_2020_2024_1.csv
  EGMS_L2a_073_0287_IW1_VV_2020_2024_1.csv


In [62]:
# ============================================================
# PROCESS EGMS CSV FILES
# ============================================================

temp_files = []

global_chunk_number = 0


for f in csv_files:
    print(f"  {f.name}")


# ============================================================
# PROCESS EACH CSV
# ============================================================

for csv_index, csv_file in enumerate(
    csv_files,
    start=1
):

    print()
    print("=" * 70)
    print(
        f"CSV FILE "
        f"{csv_index}/{len(csv_files)}"
    )
    print(csv_file.name)
    print("=" * 70)

    # --------------------------------------------------------
    # Determine number of rows and chunks
    # --------------------------------------------------------

    with open(csv_file, "rb") as f:
        feature_count = sum(1 for _ in f) - 1  # subtract header

    number_of_chunks = (
        feature_count + CHUNK_SIZE - 1
    ) // CHUNK_SIZE

    print(
        f"Total points: "
        f"{feature_count:,}"
    )

    print(
        f"Chunk size: "
        f"{CHUNK_SIZE:,}"
    )

    print(
        f"Chunks to process: "
        f"{number_of_chunks:,}"
    )

    # --------------------------------------------------------
    # Process CSV in chunks
    # --------------------------------------------------------

    for chunk_index, gdf_chunk in enumerate(
        pd.read_csv(
            csv_file,
            chunksize=CHUNK_SIZE
        ),
        start=1
    ):

        print()
        print(
            f"Chunk {chunk_index}"
        )

        print_memory(
            "after reading chunk"
        )

        print(
            f"Loaded: "
            f"{len(gdf_chunk):,} points"
        )

        # ----------------------------------------------------
        # CALCULATE VELOCITIES
        # ----------------------------------------------------

        gdf_chunk = add_velocity(
            gdf_chunk,
            intervals
        )

        print_memory(
            "after velocity calculation"
        )

        # ----------------------------------------------------
        # CREATE GEOMETRY
        # ----------------------------------------------------

        gdf_chunk = gpd.GeoDataFrame(
            gdf_chunk,
            geometry=gpd.points_from_xy(
                gdf_chunk["longitude"],
                gdf_chunk["latitude"]
            ),
            crs="EPSG:4326"
        )

        print(
            "Geometry created: "
            f"{gdf_chunk.geometry.notna().sum():,} points"
        )

        print_memory(
            "after geometry creation"
        )

        # ----------------------------------------------------
        # WRITE TEMPORARY GEOPARQUET
        # ----------------------------------------------------

        global_chunk_number += 1

        temp_file = (
            temp_dir /
            f"part_{global_chunk_number:05d}.parquet"
        )

        gdf_chunk.to_parquet(
            temp_file,
            index=False,
            compression="zstd"
        )

        temp_files.append(
            temp_file
        )

        print(
            f"Written GeoParquet: "
            f"{temp_file.name}"
        )

        print_memory(
            "after writing chunk"
        )

        # ----------------------------------------------------
        # RELEASE CHUNK
        # ----------------------------------------------------

        del gdf_chunk

        gc.collect()

        print_memory(
            "after chunk cleanup"
        )

  EGMS_L2a_073_0285_IW1_VV_2020_2024_1.csv
  EGMS_L2a_073_0287_IW1_VV_2020_2024_1.csv

CSV FILE 1/2
EGMS_L2a_073_0285_IW1_VV_2020_2024_1.csv

Chunk 1
[MEMORY] after reading chunk: 0.76 GB RSS
Loaded: 20,000 points
    Processing 2022-01-01 → 2024-12-31
      Using 92 observations (20220101 → 20241228)
    Processing 2020-01-01 → 2024-12-31
      Using 208 observations (20200106 → 20241228)
[MEMORY] after velocity calculation: 0.77 GB RSS
Geometry created: 20,000 points
[MEMORY] after geometry creation: 0.81 GB RSS
Written GeoParquet: part_00001.parquet
[MEMORY] after writing chunk: 0.81 GB RSS
[MEMORY] after chunk cleanup: 0.78 GB RSS

Chunk 2
[MEMORY] after reading chunk: 0.78 GB RSS
Loaded: 20,000 points
    Processing 2022-01-01 → 2024-12-31
      Using 92 observations (20220101 → 20241228)
    Processing 2020-01-01 → 2024-12-31
      Using 208 observations (20200106 → 20241228)
[MEMORY] after velocity calculation: 0.78 GB RSS
Geometry created: 20,000 points
[MEMORY] after geometry 

In [63]:
# ============================================================
# CHECK TEMP PARQUET SCHEMAS
# ============================================================

for temp_file in temp_files:

    parquet_file = pq.ParquetFile(temp_file)

    columns = parquet_file.schema_arrow.names

    print(
        f"{temp_file.name}: "
        f"{len(columns)} columns"
    )

part_00001.parquet: 238 columns
part_00002.parquet: 238 columns
part_00003.parquet: 238 columns
part_00004.parquet: 238 columns
part_00005.parquet: 238 columns
part_00006.parquet: 238 columns
part_00007.parquet: 238 columns
part_00008.parquet: 238 columns
part_00009.parquet: 238 columns
part_00010.parquet: 238 columns
part_00011.parquet: 238 columns
part_00012.parquet: 238 columns
part_00013.parquet: 238 columns
part_00014.parquet: 238 columns
part_00015.parquet: 238 columns
part_00016.parquet: 238 columns
part_00017.parquet: 238 columns
part_00018.parquet: 238 columns
part_00019.parquet: 238 columns
part_00020.parquet: 238 columns
part_00021.parquet: 239 columns


In [65]:
# ============================================================
# FIND COLUMN DIFFERENCES
# ============================================================

schemas = {}

for temp_file in temp_files:

    parquet_file = pq.ParquetFile(temp_file)

    schemas[temp_file.name] = set(
        parquet_file.schema_arrow.names
    )


all_columns = set().union(
    *schemas.values()
)

for filename, columns in schemas.items():

    missing = all_columns - columns

    if missing:
        print()
        print(filename)
        print("Missing columns:")
        print(sorted(missing))


part_00001.parquet
Missing columns:
['20211126']

part_00002.parquet
Missing columns:
['20211126']

part_00003.parquet
Missing columns:
['20211126']

part_00004.parquet
Missing columns:
['20211126']

part_00005.parquet
Missing columns:
['20211126']

part_00006.parquet
Missing columns:
['20211126']

part_00007.parquet
Missing columns:
['20211126']

part_00008.parquet
Missing columns:
['20211126']

part_00009.parquet
Missing columns:
['20211126']

part_00010.parquet
Missing columns:
['20211126']

part_00011.parquet
Missing columns:
['20211126']

part_00012.parquet
Missing columns:
['20211126']

part_00013.parquet
Missing columns:
['20211126']

part_00014.parquet
Missing columns:
['20211126']

part_00015.parquet
Missing columns:
['20211126']

part_00016.parquet
Missing columns:
['20211126']

part_00017.parquet
Missing columns:
['20211126']

part_00018.parquet
Missing columns:
['20211126']

part_00019.parquet
Missing columns:
['20211126']

part_00020.parquet
Missing columns:
['20211126']


In [66]:
# ============================================================
# DETERMINE UNION OF ALL COLUMNS
# ============================================================

all_columns = []

for temp_file in temp_files:

    parquet_file = pq.ParquetFile(temp_file)

    for column in parquet_file.schema_arrow.names:

        if column not in all_columns:
            all_columns.append(column)

print(
    f"Total columns in final dataset: "
    f"{len(all_columns)}"
)


Total columns in final dataset: 239


In [67]:
# ============================================================
# COMBINE ALL TEMPORARY PARQUETS
# ============================================================

print()
print("=" * 70)
print("COMBINING TEMPORARY PARQUETS")
print("=" * 70)

print(
    f"Temporary files: {len(temp_files)}"
)

print(
    f"Final output: {final_output}"
)


# ------------------------------------------------------------
# Establish final column order
# ------------------------------------------------------------

all_columns = []

for temp_file in temp_files:

    parquet_file = pq.ParquetFile(temp_file)

    for column in parquet_file.schema_arrow.names:

        if column not in all_columns:
            all_columns.append(column)


print(
    f"Final number of columns: "
    f"{len(all_columns)}"
)


# ------------------------------------------------------------
# Use first file as base schema
# ------------------------------------------------------------

first_table = pq.read_table(
    temp_files[0]
)

# Add missing columns to first table
for column in all_columns:

    if column not in first_table.column_names:

        first_table = first_table.append_column(
            column,
            pa.nulls(
                first_table.num_rows,
                type=pa.float64()
            )
        )

# Reorder
first_table = first_table.select(
    all_columns
)

schema = first_table.schema

del first_table

gc.collect()


# ------------------------------------------------------------
# Create writer
# ------------------------------------------------------------

writer = pq.ParquetWriter(
    final_output,
    schema,
    compression="zstd"
)


# ------------------------------------------------------------
# Append all temporary Parquets
# ------------------------------------------------------------

for i, temp_file in enumerate(
    temp_files,
    start=1
):

    print(
        f"Combining "
        f"{i}/{len(temp_files)}: "
        f"{temp_file.name}"
    )

    table = pq.read_table(
        temp_file
    )

    # --------------------------------------------------------
    # Add missing columns
    # --------------------------------------------------------

    missing_columns = [
        column
        for column in all_columns
        if column not in table.column_names
    ]

    for column in missing_columns:

        field = schema.field(
            schema.get_field_index(column)
        )

        table = table.append_column(
            column,
            pa.nulls(
                table.num_rows,
                type=field.type
            )
        )

    # --------------------------------------------------------
    # Reorder columns to match final schema
    # --------------------------------------------------------

    table = table.select(
        all_columns
    )

    writer.write_table(
        table
    )

    del table

    gc.collect()


# ------------------------------------------------------------
# Close writer
# ------------------------------------------------------------

writer.close()

print()
print("=" * 70)
print("FINAL PARQUET CREATED")
print("=" * 70)

print(final_output)

print_memory(
    "after final parquet"
)


COMBINING TEMPORARY PARQUETS
Temporary files: 21
Final output: /home/lukasscharf/projects/ADUCAT/data/EGMS/vienna/processed/EGMS_073_2020_2024.parquet
Final number of columns: 239
Combining 1/21: part_00001.parquet
Combining 2/21: part_00002.parquet
Combining 3/21: part_00003.parquet
Combining 4/21: part_00004.parquet
Combining 5/21: part_00005.parquet
Combining 6/21: part_00006.parquet
Combining 7/21: part_00007.parquet
Combining 8/21: part_00008.parquet
Combining 9/21: part_00009.parquet
Combining 10/21: part_00010.parquet
Combining 11/21: part_00011.parquet
Combining 12/21: part_00012.parquet
Combining 13/21: part_00013.parquet
Combining 14/21: part_00014.parquet
Combining 15/21: part_00015.parquet
Combining 16/21: part_00016.parquet
Combining 17/21: part_00017.parquet
Combining 18/21: part_00018.parquet
Combining 19/21: part_00019.parquet
Combining 20/21: part_00020.parquet
Combining 21/21: part_00021.parquet

FINAL PARQUET CREATED
/home/lukasscharf/projects/ADUCAT/data/EGMS/vienn

In [76]:
combined_file = Path(f"{final_output}"
)
print(combined_file)

/home/lukasscharf/projects/ADUCAT/data/EGMS/vienna/processed/EGMS_073_2020_2024.parquet


In [77]:
schema = pq.read_schema(combined_file)

print("=" * 70)
print("COMBINED PARQUET")
print("=" * 70)

print(f"Number of columns: {len(schema.names)}")

for i, column in enumerate(schema.names, start=1):
    print(f"{i:3d}: {column}")

COMBINED PARQUET
Number of columns: 239
  1: pid
  2: cluster_label
  3: mp_type
  4: latitude
  5: longitude
  6: easting
  7: northing
  8: height_ortho
  9: height_ellipse
 10: line
 11: pixel
 12: rmse_ts
 13: temporal_coherence
 14: amplitude_dispersion
 15: incidence_angle
 16: track_angle
 17: los_east
 18: los_north
 19: los_up
 20: mean_velocity
 21: mean_velocity_std
 22: acceleration
 23: acceleration_std
 24: seasonality
 25: seasonality_std
 26: 20200106
 27: 20200112
 28: 20200118
 29: 20200124
 30: 20200130
 31: 20200205
 32: 20200211
 33: 20200217
 34: 20200223
 35: 20200306
 36: 20200312
 37: 20200318
 38: 20200324
 39: 20200330
 40: 20200405
 41: 20200411
 42: 20200417
 43: 20200423
 44: 20200429
 45: 20200505
 46: 20200511
 47: 20200517
 48: 20200523
 49: 20200529
 50: 20200604
 51: 20200610
 52: 20200616
 53: 20200622
 54: 20200628
 55: 20200704
 56: 20200710
 57: 20200716
 58: 20200722
 59: 20200728
 60: 20200803
 61: 20200809
 62: 20200815
 63: 20200821
 64: 20200

### Seperate to velocity and timer series file

In [68]:
velocity_file = (
    OUTPUT_DIR /
    f"EGMS_{ORBIT}_{PRODUCT_RANGE}_without_timeseries.parquet"
)

timeseries_file = (
    OUTPUT_DIR /
    f"EGMS_{ORBIT}_{PRODUCT_RANGE}__just_timeseries.parquet"
)

print(velocity_file)
print(timeseries_file)

/home/lukasscharf/projects/ADUCAT/data/EGMS/vienna/processed/EGMS_073_2020_2024_without_timeseries.parquet
/home/lukasscharf/projects/ADUCAT/data/EGMS/vienna/processed/EGMS_073_2020_2024__just_timeseries.parquet


In [69]:
# --------------------------------------------------
# Full Parquet file
# --------------------------------------------------

pf = pq.ParquetFile(final_output)

columns = pf.schema_arrow.names

In [70]:
# --------------------------------------------------
# Identify time-series columns
# --------------------------------------------------

ts_cols = [
    c for c in columns
    if re.fullmatch(r"\d{8}", str(c))
]

# Everything else goes into spatial/velocity file
# --------------------------------------------------

velocity_columns = [
    c for c in columns
    if c not in ts_cols
]

print("Total columns:", len(columns))
print("Time-series columns:", len(ts_cols))
print("Velocity/static columns:", len(velocity_columns))

Total columns: 239
Time-series columns: 209
Velocity/static columns: 30


In [74]:
# ============================================================
# CREATE VELOCITY PARQUET
# ============================================================

print()
print("Creating velocity Parquet...")

velocity_writer = None

if velocity_file.exists():
    velocity_file.unlink()

for batch_number, batch in enumerate(
    pf.iter_batches(
        batch_size=CHUNK_SIZE,
        columns=velocity_columns
    ),
    start=1
):

    print(
        f"Velocity batch "
        f"{batch_number}: "
        f"{batch.num_rows:,} rows"
    )

    table = pa.Table.from_batches([batch])

    # --------------------------------------------------------
    # Create writer from FIRST batch only
    # --------------------------------------------------------

    if velocity_writer is None:

        velocity_writer = pq.ParquetWriter(
            velocity_file,
            table.schema,
            compression="zstd"
        )

        print(
            f"Created writer with "
            f"{len(table.schema.names)} columns"
        )

    # --------------------------------------------------------
    # Write batch
    # --------------------------------------------------------

    velocity_writer.write_table(table)

    del table
    del batch

    gc.collect()


# ------------------------------------------------------------
# Close writer
# ------------------------------------------------------------

if velocity_writer is not None:
    velocity_writer.close()
    velocity_writer = None

print(
    f"Velocity file created:\n"
    f"{velocity_file}"
)


Creating velocity Parquet...
Velocity batch 1: 20,000 rows
Created writer with 30 columns
Velocity batch 2: 20,000 rows
Velocity batch 3: 20,000 rows
Velocity batch 4: 20,000 rows
Velocity batch 5: 20,000 rows
Velocity batch 6: 20,000 rows
Velocity batch 7: 20,000 rows
Velocity batch 8: 20,000 rows
Velocity batch 9: 20,000 rows
Velocity batch 10: 20,000 rows
Velocity batch 11: 20,000 rows
Velocity batch 12: 20,000 rows
Velocity batch 13: 20,000 rows
Velocity batch 14: 20,000 rows
Velocity batch 15: 20,000 rows
Velocity batch 16: 20,000 rows
Velocity batch 17: 20,000 rows
Velocity batch 18: 20,000 rows
Velocity batch 19: 20,000 rows
Velocity batch 20: 9,387 rows
Velocity file created:
/home/lukasscharf/projects/ADUCAT/data/EGMS/vienna/processed/EGMS_073_2020_2024_without_timeseries.parquet


In [75]:
# ============================================================
# CREATE TIME-SERIES PARQUET
# ============================================================

print()
print("Creating time-series Parquet...")

timeseries_writer = None

if timeseries_file.exists():
    timeseries_file.unlink()

for batch_number, batch in enumerate(
    pf.iter_batches(
        batch_size=CHUNK_SIZE,
        columns=ts_cols
    ),
    start=1
):

    print(
        f"Time-series batch "
        f"{batch_number}: "
        f"{batch.num_rows:,} rows"
    )

    table = pa.Table.from_batches([batch])

    # --------------------------------------------------------
    # Create writer from FIRST batch only
    # --------------------------------------------------------

    if timeseries_writer is None:

        timeseries_writer = pq.ParquetWriter(
            timeseries_file,
            table.schema,
            compression="zstd"
        )

        print(
            f"Created writer with "
            f"{len(table.schema.names)} columns"
        )

    # --------------------------------------------------------
    # Write batch
    # --------------------------------------------------------

    timeseries_writer.write_table(table)

    del table
    del batch

    gc.collect()


# ------------------------------------------------------------
# Close writer
# ------------------------------------------------------------

if timeseries_writer is not None:
    timeseries_writer.close()
    timeseries_writer = None

print(
    f"Time-series file created:\n"
    f"{timeseries_file}"
)


Creating time-series Parquet...
Time-series batch 1: 20,000 rows
Created writer with 209 columns
Time-series batch 2: 20,000 rows
Time-series batch 3: 20,000 rows
Time-series batch 4: 20,000 rows
Time-series batch 5: 20,000 rows
Time-series batch 6: 20,000 rows
Time-series batch 7: 20,000 rows
Time-series batch 8: 20,000 rows
Time-series batch 9: 20,000 rows
Time-series batch 10: 20,000 rows
Time-series batch 11: 20,000 rows
Time-series batch 12: 20,000 rows
Time-series batch 13: 20,000 rows
Time-series batch 14: 20,000 rows
Time-series batch 15: 20,000 rows
Time-series batch 16: 20,000 rows
Time-series batch 17: 20,000 rows
Time-series batch 18: 20,000 rows
Time-series batch 19: 20,000 rows
Time-series batch 20: 9,387 rows
Time-series file created:
/home/lukasscharf/projects/ADUCAT/data/EGMS/vienna/processed/EGMS_073_2020_2024__just_timeseries.parquet


In [72]:
# ============================================================
# REMOVE TEMPORARY FILES
# ============================================================

if final_output.exists():

    print()
    print("Removing temporary files...")

    shutil.rmtree(temp_dir)

    print(
        f"Removed temporary directory: "
        f"{temp_dir}"
    )

else:

    print()
    print(
        "WARNING: Final output does not exist."
    )

    print(
        "Temporary files were NOT deleted."
    )


Removing temporary files...
Removed temporary directory: /home/lukasscharf/projects/ADUCAT/data/EGMS/vienna/processed/temp_chunks
